In [45]:
from __future__ import annotations

import ast
import math
import re
import textwrap
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

In [46]:
@dataclass(frozen=True)
class ProjectConfig:
    language: str = "python"
    dataset_name: str = "code_search_net"
    dataset_config: str = "python"
    split: str = "train"
    sample_size: int = 2_000
    embedding_model_name: str = "microsoft/codebert-base"
    summarizer_model_name: str = "Salesforce/codet5-small"
    max_tokens: int = 256
    embedding_batch_size: int = 16
    retrieval_k: int = 5
    random_state: int = 42


CONFIG = ProjectConfig()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [47]:
def load_codesearchnet_subset(config: ProjectConfig) -> pd.DataFrame:
    try:
        from datasets import load_dataset

        dataset = load_dataset(
            config.dataset_name,
            config.dataset_config,
            split=f"{config.split}[:{config.sample_size}]",
        )
        frame = dataset.to_pandas()
        code_col = "func_code_string" if "func_code_string" in frame.columns else "code"
        doc_col = "func_documentation_string" if "func_documentation_string" in frame.columns else "docstring"
        output = pd.DataFrame(
            {
                "code": frame[code_col].fillna("").astype(str),
                "docstring": frame[doc_col].fillna("").astype(str),
                "language": config.language,
                "source": "CodeSearchNet",
            }
        )
        return output
    except Exception as exc:
        print(f"Could not load CodeSearchNet, using fallback examples. Reason: {exc}")
        return pd.DataFrame(
            [
                {
                    "code": "def binary_search(arr, target):\n    lo, hi = 0, len(arr)-1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        if arr[mid] < target:\n            lo = mid + 1\n        else:\n            hi = mid - 1\n    return -1",
                    "docstring": "Return the index of target in a sorted list using binary search.",
                    "language": "python",
                    "source": "fallback",
                },
                {
                    "code": "def has_duplicate(nums):\n    seen = set()\n    for n in nums:\n        if n in seen:\n            return True\n        seen.add(n)\n    return False",
                    "docstring": "Check whether a list contains duplicate values.",
                    "language": "python",
                    "source": "fallback",
                },
                {
                    "code": "def bubble_sort(values):\n    n = len(values)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if values[j] > values[j+1]:\n                values[j], values[j+1] = values[j+1], values[j]\n    return values",
                    "docstring": "Sort values in ascending order using bubble sort.",
                    "language": "python",
                    "source": "fallback",
                },
            ]
        )

In [48]:
raw_df = load_codesearchnet_subset(CONFIG)
print(raw_df.shape)
raw_df.head()

Could not load CodeSearchNet, using fallback examples. Reason: Invalid HF URI 'hf://datasets/code_search_net@bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/.huggingface.yaml'. Repository id must be 'namespace/name', got 'code_search_net'.
(3, 4)


,code,docstring,language,source
0,"def binary_search(arr, target):\n lo, hi = ...",Return the index of target in a sorted list us...,python,fallback
1,def has_duplicate(nums):\n seen = set()\n ...,Check whether a list contains duplicate values.,python,fallback
2,def bubble_sort(values):\n n = len(values)\...,Sort values in ascending order using bubble sort.,python,fallback


In [49]:
def strip_python_comments_and_docstrings(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return re.sub(r"#.*", "", code)

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):
            if (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(getattr(node.body[0], "value", None), ast.Constant)
                and isinstance(node.body[0].value.value, str)
            ):
                node.body[0] = ast.Pass()
    try:
        return ast.unparse(tree)
    except Exception:
        return code

In [50]:
def normalize_code(code: str) -> str:
    code = textwrap.dedent(str(code)).strip()
    code = strip_python_comments_and_docstrings(code)
    code = re.sub(r"\n{3,}", "\n\n", code)
    code = "\n".join(line.rstrip() for line in code.splitlines())
    return code.strip()


def prepare_dataset(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    prepared["clean_code"] = prepared["code"].map(normalize_code)
    prepared["docstring"] = prepared["docstring"].fillna("").astype(str).str.strip()
    prepared = prepared[prepared["clean_code"].str.len() > 20].drop_duplicates("clean_code")
    prepared = prepared.reset_index(drop=True)
    return prepared

In [51]:
code_df = prepare_dataset(raw_df)
assert not code_df.empty, "Dataset is empty after preprocessing."
print(code_df.shape)
print(code_df.loc[0, "clean_code"][:500])

(3, 5)
def binary_search(arr, target):
    lo, hi = (0, len(arr) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


In [52]:
from transformers import AutoModel, AutoTokenizer


class CodeEmbedder:
    def __init__(self, model_name: str, device: str = DEVICE, max_tokens: int = 256):
        self.model_name = model_name
        self.device = device
        self.max_tokens = max_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
    @torch.inference_mode()
    def encode(self, texts: Sequence[str], batch_size: int = 16) -> np.ndarray:
        vectors: List[np.ndarray] = []
        for start in tqdm(range(0, len(texts), batch_size), desc="Embedding code"):
            batch = list(texts[start : start + batch_size])
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_tokens,
                return_tensors="pt",
            ).to(self.device)
            output = self.model(**encoded)
            mask = encoded["attention_mask"].unsqueeze(-1)
            token_embeddings = output.last_hidden_state * mask
            summed = token_embeddings.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1)
            pooled = summed / counts
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            vectors.append(pooled.cpu().numpy().astype("float32"))
        return np.vstack(vectors)

In [53]:
embedder = CodeEmbedder(CONFIG.embedding_model_name, max_tokens=CONFIG.max_tokens)
code_embeddings = embedder.encode(
    code_df["clean_code"].tolist(),
    batch_size=CONFIG.embedding_batch_size,
)
print(code_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding code:   0%|          | 0/1 [00:00<?, ?it/s]

(3, 768)


In [54]:
class VectorIndex:
    def __init__(self, vectors: np.ndarray):
        self.vectors = vectors.astype("float32")
        self.faiss_index = None
        try:
            import faiss

            self.faiss_index = faiss.IndexFlatIP(self.vectors.shape[1])
            self.faiss_index.add(self.vectors)
            self.backend = "faiss"
        except Exception as exc:
            print(f"FAISS unavailable, using sklearn cosine search. Reason: {exc}")
            self.backend = "sklearn"

    def search(self, query_vectors: np.ndarray, k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        if self.backend == "faiss":
            scores, indices = self.faiss_index.search(query_vectors, k)
            return scores, indices
        scores = cosine_similarity(query_vectors, self.vectors)
        indices = np.argsort(-scores, axis=1)[:, :k]
        top_scores = np.take_along_axis(scores, indices, axis=1)
        return top_scores, indices

In [55]:
vector_index = VectorIndex(code_embeddings)
print(f"Vector search backend: {vector_index.backend}")

FAISS unavailable, using sklearn cosine search. Reason: No module named 'faiss'
Vector search backend: sklearn


In [56]:
def retrieve_similar_code(
    code: str,
    embedder: CodeEmbedder,
    index: VectorIndex,
    corpus: pd.DataFrame,
    k: int = 5,
) -> List[Dict[str, Any]]:
    clean = normalize_code(code)
    query_vector = embedder.encode([clean], batch_size=1)
    scores, indices = index.search(query_vector, k=k)
    results: List[Dict[str, Any]] = []
    for score, idx in zip(scores[0], indices[0]):
        row = corpus.iloc[int(idx)]
        results.append(
            {
                "score": float(score),
                "code": row["clean_code"],
                "docstring": row["docstring"],
                "source": row["source"],
            }
        )
    return results

In [57]:
example_code = """
def find_item(items, x):
    for i, value in enumerate(items):
        if value == x:
            return i
    return -1
"""

retrieved = retrieve_similar_code(example_code, embedder, vector_index, code_df, k=CONFIG.retrieval_k)
pd.DataFrame([{k: v for k, v in item.items() if k != "code"} for item in retrieved])

Embedding code:   0%|          | 0/1 [00:00<?, ?it/s]

,score,docstring,source
0,0.993006,Check whether a list contains duplicate values.,fallback
1,0.984937,Sort values in ascending order using bubble sort.,fallback
2,0.979848,Return the index of target in a sorted list us...,fallback


In [58]:
from transformers import AutoModelForSeq2SeqLM

class CodeSummarizer:
    def __init__(self, model_name: str, device: str = DEVICE):
        self.available = False
        self.device = device
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
            self.model.eval()
            self.available = True
        except Exception as exc:
            print(f"Transformer summarizer unavailable, using fallback summary. Reason: {exc}")

    @torch.inference_mode()
    def summarize(self, code: str, retrieved_context: Optional[List[Dict[str, Any]]] = None) -> str:
        clean = normalize_code(code)
        if self.available:
            prompt = "summarize: " + clean
            encoded = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=384,
            ).to(self.device)
            generated = self.model.generate(
                **encoded,
                max_length=80,
                num_beams=4,
                early_stopping=True,
            )
            return self.tokenizer.decode(generated[0], skip_special_tokens=True).strip()
        return fallback_code_summary(clean, retrieved_context or [])

In [59]:
def fallback_code_summary(code: str, retrieved_context: List[Dict[str, Any]]) -> str:
    try:
        tree = ast.parse(code)
        functions = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
        if functions:
            fn = functions[0]
            name = fn.name.replace("_", " ")
            args = [arg.arg for arg in fn.args.args]
            return f"Defines `{fn.name}` to perform {name} using inputs: {', '.join(args) or 'none'}."
    except SyntaxError:
        pass
    for item in retrieved_context:
        if item.get("docstring"):
            return f"Likely purpose based on similar code: {item['docstring']}"
    return "The code could not be summarized reliably from syntax or retrieved examples."


summarizer = CodeSummarizer(CONFIG.summarizer_model_name)
print(summarizer.summarize(example_code, retrieved))

Transformer summarizer unavailable, using fallback summary. Reason: Input must be a List[Union[str, AddedToken]]
Defines `find_item` to perform find item using inputs: items, x.


In [60]:
COMPLEXITY_EXAMPLES = [
    ("def f(a):\n    return a[0]", "O(1)", "O(1)"),
    ("def f(a):\n    for x in a:\n        print(x)", "O(n)", "O(1)"),
    ("def f(a):\n    s=set()\n    for x in a:\n        s.add(x)\n    return s", "O(n)", "O(n)"),
    ("def f(a):\n    for x in a:\n        for y in a:\n            print(x,y)", "O(n^2)", "O(1)"),
    ("def f(a):\n    if not a: return None\n    return f(a[1:])", "O(n^2)", "O(n)"),
    ("def f(a, target):\n    lo=0; hi=len(a)-1\n    while lo<=hi:\n        mid=(lo+hi)//2\n        if a[mid]==target: return mid\n        if a[mid]<target: lo=mid+1\n        else: hi=mid-1\n    return -1", "O(log n)", "O(1)"),
    ("def f(a):\n    if len(a)<=1: return a\n    mid=len(a)//2\n    return f(a[:mid]) + f(a[mid:])", "O(n log n)", "O(n log n)"),
]

In [61]:
def ast_complexity_features(code: str) -> Dict[str, float]:
    features = {
        "for_loops": 0,
        "while_loops": 0,
        "nested_loop_pairs": 0,
        "recursive_calls": 0,
        "list_set_dict_comps": 0,
        "allocating_literals": 0,
        "max_loop_depth": 0,
        "slices": 0,
    }
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return features

    function_names = {node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)}
    def visit(node: ast.AST, loop_depth: int = 0) -> None:
        is_loop = isinstance(node, (ast.For, ast.While))
        next_depth = loop_depth + 1 if is_loop else loop_depth
        if isinstance(node, ast.For):
            features["for_loops"] += 1
        if isinstance(node, ast.While):
            features["while_loops"] += 1
        if is_loop and loop_depth > 0:
            features["nested_loop_pairs"] += 1
        features["max_loop_depth"] = max(features["max_loop_depth"], next_depth)
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in function_names:
            features["recursive_calls"] += 1
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            features["list_set_dict_comps"] += 1
        if isinstance(node, (ast.List, ast.Set, ast.Dict)):
            features["allocating_literals"] += 1
        if isinstance(node, ast.Slice):
            features["slices"] += 1
        for child in ast.iter_child_nodes(node):
            visit(child, next_depth)

    visit(tree)
    return features

In [62]:
class ComplexityEstimator:
    def __init__(self):
        self.time_encoder = LabelEncoder()
        self.space_encoder = LabelEncoder()
        self.time_model = RandomForestClassifier(n_estimators=200, random_state=CONFIG.random_state)
        self.space_model = RandomForestClassifier(n_estimators=200, random_state=CONFIG.random_state)
        self.feature_names = list(ast_complexity_features(COMPLEXITY_EXAMPLES[0][0]).keys())

    def _matrix(self, codes: Sequence[str]) -> np.ndarray:
        rows = [ast_complexity_features(normalize_code(code)) for code in codes]
        return np.array([[row[name] for name in self.feature_names] for row in rows], dtype="float32")

    def fit(self, examples: Sequence[Tuple[str, str, str]]) -> "ComplexityEstimator":
        codes = [item[0] for item in examples]
        time_labels = self.time_encoder.fit_transform([item[1] for item in examples])
        space_labels = self.space_encoder.fit_transform([item[2] for item in examples])
        x = self._matrix(codes)
        self.time_model.fit(x, time_labels)
        self.space_model.fit(x, space_labels)
        return self

    def predict(self, code: str) -> Dict[str, Any]:
        x = self._matrix([code])
        time_probs = self.time_model.predict_proba(x)[0]
        space_probs = self.space_model.predict_proba(x)[0]
        time_idx = int(np.argmax(time_probs))
        space_idx = int(np.argmax(space_probs))
        return {
            "time_complexity": self.time_encoder.inverse_transform([time_idx])[0],
            "space_complexity": self.space_encoder.inverse_transform([space_idx])[0],
            "time_confidence": float(time_probs[time_idx]),
            "space_confidence": float(space_probs[space_idx]),
            "features": ast_complexity_features(normalize_code(code)),
        }

In [63]:
complexity_estimator = ComplexityEstimator().fit(COMPLEXITY_EXAMPLES)
complexity_estimator.predict(example_code)

{'time_complexity': np.str_('O(n)'),
 'space_complexity': np.str_('O(1)'),
 'time_confidence': 0.9,
 'space_confidence': 0.5519999999999999,
 'features': {'for_loops': 1,
  'while_loops': 0,
  'nested_loop_pairs': 0,
  'recursive_calls': 0,
  'list_set_dict_comps': 0,
  'allocating_literals': 0,
  'max_loop_depth': 1,
  'slices': 0}}

In [64]:
def detect_bug_patterns(code: str) -> List[Dict[str, str]]:
    clean = normalize_code(code)
    findings: List[Dict[str, str]] = []
    try:
        tree = ast.parse(clean)
    except SyntaxError as exc:
        return [{"severity": "high", "issue": "Syntax error", "detail": str(exc)}]

    for node in ast.walk(tree):
        if isinstance(node, ast.ExceptHandler) and node.type is None:
            findings.append(
                {
                    "severity": "medium",
                    "issue": "Bare except",
                    "detail": "Catches all exceptions and can hide programming errors.",
                }
            )
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "eval":
            findings.append(
                {
                    "severity": "high",
                    "issue": "Use of eval",
                    "detail": "Executing dynamic input can create security vulnerabilities.",
                }
            )
        if isinstance(node, ast.Compare):
            for op in node.ops:
                if isinstance(op, (ast.Is, ast.IsNot)):
                    comparators = [node.left, *node.comparators]
                    if not any(isinstance(item, ast.Constant) and item.value is None for item in comparators):
                        findings.append(
                            {
                                "severity": "medium",
                                "issue": "Identity comparison",
                                "detail": "`is` should usually be reserved for None/singletons, not value equality.",
                            }
                        )
        if isinstance(node, ast.FunctionDef):
            for default in node.args.defaults:
                if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                    findings.append(
                        {
                            "severity": "high",
                            "issue": "Mutable default argument",
                            "detail": "Mutable defaults are shared across calls and can cause state leakage.",
                        }
                    )
    return findings

In [65]:
detect_bug_patterns("def f(x=[]):\n    x.append(1)\n    return x")


[{'severity': 'high',
  'issue': 'Mutable default argument',
  'detail': 'Mutable defaults are shared across calls and can cause state leakage.'}]

In [66]:
def analyze_quality(code: str) -> List[Dict[str, str]]:
    clean = normalize_code(code)
    issues: List[Dict[str, str]] = []
    lines = clean.splitlines()
    if len(lines) > 80:
        issues.append({"issue": "Large function", "detail": "Consider splitting into smaller units."})
    if any(len(line) > 100 for line in lines):
        issues.append({"issue": "Long line", "detail": "Keep lines readable and formatter-friendly."})
    try:
        tree = ast.parse(clean)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef) and not ast.get_docstring(node):
                issues.append({"issue": "Missing function docstring", "detail": f"`{node.name}` has no docstring."})
    except SyntaxError:
        pass
    return issues

In [67]:
def suggest_optimizations(
    code: str,
    complexity: Dict[str, Any],
    retrieved_context: List[Dict[str, Any]],
) -> List[str]:
    suggestions: List[str] = []
    features = complexity["features"]
    if features["max_loop_depth"] >= 2:
        suggestions.append("Nested loops detected. Check whether a set, dictionary, sorting step, or two-pointer strategy can reduce repeated scanning.")
    if features["slices"] and features["recursive_calls"]:
        suggestions.append("Recursive slicing can copy data repeatedly. Prefer index boundaries or memoization where appropriate.")
    if complexity["time_complexity"] in {"O(n^2)", "O(n^3)"}:
        suggestions.append("The estimated time complexity is high. Compare against retrieved implementations for lower-complexity approaches.")
    best_docs = [item["docstring"] for item in retrieved_context if item.get("docstring")]
    if best_docs:
        suggestions.append(f"Nearest retrieved implementation says: {best_docs[0][:180]}")
    return suggestions or ["No obvious optimization pattern was detected."]

In [68]:
print(analyze_quality(example_code))
print(suggest_optimizations(example_code, complexity_estimator.predict(example_code), retrieved))


[{'issue': 'Missing function docstring', 'detail': '`find_item` has no docstring.'}]
['Nearest retrieved implementation says: Check whether a list contains duplicate values.']


In [69]:
def propose_improved_code(code: str, bug_findings: List[Dict[str, str]]) -> Optional[str]:
    clean = normalize_code(code)
    if any(item["issue"] == "Mutable default argument" for item in bug_findings):
        fixed = re.sub(r"def (\w+)\((\w+)=\[\]\):", r"def \1(\2=None):", clean)
        fixed = re.sub(r"(\n\s*)(\w+)\.append", r"\1if \2 is None:\1    \2 = []\1\2.append", fixed, count=1)
        return fixed
    return None


bad_code = "def append_one(items=[]):\n    items.append(1)\n    return items"
propose_improved_code(bad_code, detect_bug_patterns(bad_code))

'def append_one(items=None):\n    if items is None:\n        items = []\n    items.append(1)\n    return items'

In [70]:
def review_code(code: str) -> Dict[str, Any]:
    clean = normalize_code(code)
    similar = retrieve_similar_code(clean, embedder, vector_index, code_df, k=CONFIG.retrieval_k)
    summary = summarizer.summarize(clean, similar)
    complexity = complexity_estimator.predict(clean)
    bugs = detect_bug_patterns(clean)
    quality = analyze_quality(clean)
    optimizations = suggest_optimizations(clean, complexity, similar)
    improved = propose_improved_code(clean, bugs)

    return {
        "summary": summary,
        "time_complexity": {
            "estimate": complexity["time_complexity"],
            "confidence": round(complexity["time_confidence"], 3),
        },
        "space_complexity": {
            "estimate": complexity["space_complexity"],
            "confidence": round(complexity["space_confidence"], 3),
        },
        "possible_bugs": bugs,
        "code_quality_issues": quality,
        "optimization_suggestions": optimizations,
        "improved_code": improved,
        "similar_code_retrieved": [
            {
                "score": round(item["score"], 3),
                "docstring": item["docstring"],
                "source": item["source"],
                "code_preview": item["code"][:500],
            }
            for item in similar
        ],
    }

In [71]:
report = review_code(example_code)
report

Embedding code:   0%|          | 0/1 [00:00<?, ?it/s]

{'summary': 'Defines `find_item` to perform find item using inputs: items, x.',
 'time_complexity': {'estimate': np.str_('O(n)'), 'confidence': 0.9},
 'space_complexity': {'estimate': np.str_('O(1)'), 'confidence': 0.552},
 'possible_bugs': [],
 'code_quality_issues': [{'issue': 'Missing function docstring',
   'detail': '`find_item` has no docstring.'}],
 'optimization_suggestions': ['Nearest retrieved implementation says: Check whether a list contains duplicate values.'],
 'improved_code': None,
 'similar_code_retrieved': [{'score': 0.993,
   'docstring': 'Check whether a list contains duplicate values.',
   'source': 'fallback',
   'code_preview': 'def has_duplicate(nums):\n    seen = set()\n    for n in nums:\n        if n in seen:\n            return True\n        seen.add(n)\n    return False'},
  {'score': 0.985,
   'docstring': 'Sort values in ascending order using bubble sort.',
   'source': 'fallback',
   'code_preview': 'def bubble_sort(values):\n    n = len(values)\n    for

In [72]:
def print_review(report: Dict[str, Any]) -> None:
    print("SUMMARY")
    print(report["summary"])
    print("\nCOMPLEXITY")
    print(f"Time: {report['time_complexity']['estimate']} (confidence {report['time_complexity']['confidence']})")
    print(f"Space: {report['space_complexity']['estimate']} (confidence {report['space_complexity']['confidence']})")
    print("\nPOSSIBLE BUGS")
    for item in report["possible_bugs"] or [{"issue": "None detected", "detail": ""}]:
        print(f"- [{item.get('severity', 'info')}] {item['issue']}: {item.get('detail', '')}")
    print("\nCODE QUALITY")
    for item in report["code_quality_issues"] or [{"issue": "No major quality issues", "detail": ""}]:
        print(f"- {item['issue']}: {item.get('detail', '')}")
    print("\nOPTIMIZATION SUGGESTIONS")
    for item in report["optimization_suggestions"]:
        print(f"- {item}")
    print("\nSIMILAR CODE RETRIEVED")
    for item in report["similar_code_retrieved"]:
        print(f"- score={item['score']} | {item['docstring'][:120]}")
    if report["improved_code"]:
        print("\nIMPROVED CODE")
        print(report["improved_code"])


print_review(report)

SUMMARY
Defines `find_item` to perform find item using inputs: items, x.

COMPLEXITY
Time: O(n) (confidence 0.9)
Space: O(1) (confidence 0.552)

POSSIBLE BUGS
- [info] None detected: 

CODE QUALITY
- Missing function docstring: `find_item` has no docstring.

OPTIMIZATION SUGGESTIONS
- Nearest retrieved implementation says: Check whether a list contains duplicate values.

SIMILAR CODE RETRIEVED
- score=0.993 | Check whether a list contains duplicate values.
- score=0.985 | Sort values in ascending order using bubble sort.
- score=0.98 | Return the index of target in a sorted list using binary search.
